# 15. WiLabV2Xsim: reproducing "Improving NR-V2X Autonomous Mode Through Resource Re-Evaluation"

V. Todisco, Z. Wu, A. Bazzi, *IEEE Vehicular Networking Conference (VNC)* 2024
(`docs/Improving_NR-V2X_Autonomous_Mode_Through_Resource_Re-Evaluation.pdf`). The paper proposes
**Double Initial Transmission (DIT)**: after every SPS reselection the first packet is sent on
two resources; the first one is released, the second one is kept periodically. The first
transmission announces the second resource, so a UE that picked the same second resource
can see it, through re-evaluation and pre-emption, before the collision repeats every period.

This notebook reproduces the paper's experiment (its Figs. 3 and 4) with WiLabV2Xsim run in
Octave, then compares WiLabV2Xsim with MoReV2X and 5G-LENA.

**Where everything is**

| Path | What |
|---|---|
| `wilabv2xsim/setup.sh` | installs micromamba and Octave 10.3 (conda-forge) inside `wilabv2xsim/`, clones WiLabV2Xsim `main` at 612bf26 and applies the patch |
| `wilabv2xsim/patches/wilabv2xsim-octave-dit.patch` | every change to WiLabV2Xsim: the Octave port and DIT with pre-emption |
| `wilabv2xsim/octave_compat/` | `db2pow`, `pow2db`, `unifrnd`, `exprnd`, `roundDigits` (MATLAB toolbox functions Octave lacks) |
| `wilabv2xsim/experiment/dit_paper.cfg` | the paper's Table II |
| `wilabv2xsim/experiment/runs.txt`, `run_one.sh`, `run_sweep.sh`, `status.sh`, `monitor.sh` | the sweep |
| `wilabv2xsim/results/<scheme>_rho<density>_s<seed>/` | WiLabV2Xsim output per run |

In [ ]:
import os, sys, json, subprocess
sys.path.insert(0, os.path.abspath("../../tools"))
sys.path.insert(0, os.path.abspath("../../tools/qos"))
import morev2x_results as mr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

mr.apply_chart_style()
os.chdir(mr.ROOT)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

import glob
RES = os.path.join(mr.ROOT, "wilabv2xsim", "results")
SCHEMES = [   # key, paper legend, colour, line style (as in the paper's figures)
    ("sps_noret_noreeval", "SPS w/o ret, no Re-Eval",     "#E69F00", "-"),
    ("sps_noret_reeval",   "SPS w/o ret, std Re-Eval",    "#0072B2", ":"),
    ("sps_ret_noreeval",   "SPS w/ ret, no Re-Eval",      "#1f1e1c", ":"),
    ("sps_ret_reeval",     "SPS w/ ret, std Re-Eval",     "#D55E00", ":"),
    ("dit_noreeval",       "DIT w/o Re-Eval, no Pre-Emp", "#009E73", "-"),
    ("dit_reeval_preempt", "DIT w/ Re-Eval and Pre-Emp",  "#882255", "-"),
]
LABEL = {k: l for k, l, _, _ in SCHEMES}
T95 = {2: 12.71, 3: 4.30, 4: 3.18, 5: 2.78}

def runs():
    rows = []
    for d in sorted(glob.glob(os.path.join(RES, "*_rho*_s*"))):
        name = os.path.basename(d)
        scheme, rest = name.rsplit("_rho", 1)
        rho, seed = rest.split("_s")
        rows.append({"dir": d, "scheme": scheme, "rho": int(rho), "seed": int(seed),
                     "done": os.path.exists(os.path.join(d, "done"))})
    return pd.DataFrame(rows)

R = runs()
print("%d runs found, %d complete" % (len(R), R.done.sum()))
R = R[R.done]
R.groupby(["rho", "scheme"]).seed.count().unstack("rho")

## 1. Setup

### 1.1 The paper's settings (Table II) and how they are set here

| Paper | Value | WiLabV2Xsim parameter |
|---|---|---|
| Road layout | highway, 3 + 3 lanes, 2 km | `TypeOfScenario ETSI-Highway`, `Nlanes 3`, `roadLength 2000` |
| Density | 50 and 200 vehicles/km (both directions) | `rho` |
| Speed | 70 ± 7 km/h | `vMean 70`, `vStDev 7` |
| Channel, bandwidth | 5.9 GHz, 20 MHz | `BwMHz 20` |
| Power, antenna gain, noise figure | 23 dBm, 3 dBi each side, 9 dB | `Ptx_dBm`, `Gt_dB`, `Gr_dB`, `F_dB` |
| Propagation | WINNER+ B1, shadowing 3 dB, decorrelation 25 m | `channelModel 0` (B1 uses 25 m), `stdDevShadowLOS_dB 3` |
| Traffic | 1000 B every 100 ms | `beaconSizeBytes 1000`, `allocationPeriod 0.1` |
| MCS, SINR threshold | MCS 5 (QPSK, R 0.3), 3.6 dB | `MCS_NR 5`, `sinrThresholdCV2X_LOS 3.6` |
| Subchannels, SCS | 10 × 10 PRB, 15 kHz (1 ms slots) | `sizeSubchannel 10`, `SCS_NR 15` |
| Keep probability, RSRP threshold | 0, −126 dBm | `probResKeep 0`, `powerThresholdAutonomous -126` |
| T1, T2, T3 | 1 ms, 100 ms, 3 ms | `T1autonomousMode 1`, `T2autonomousMode 100`, T3 = 3 slots (fixed in the code) |
| Metrics | PRR vs distance; WBSP within 150 m | `printPacketReceptionRatio` (10 m bins), `printWirelessBlindSpotProb` with `Raw [150 200]` |

At MCS 5 a 1000 B packet takes all ten subchannels, so every slot holds one packet: the channel
offers 100 resources per 100 ms.

**Not given in the paper, chosen here:** simulated time (60 s at 50 vehicles/km, 20 s at 200
vehicles/km, which is about 16 times the work per simulated second) and 3 seeds per point. Every
run counts from its start, including the initial random allocation.

### 1.2 Schemes (the six curves of the paper)

| Scheme | Copies per packet | Re-evaluation | DIT | Pre-emption |
|---|---|---|---|---|
| SPS w/o ret, no Re-Eval | 1 | no | no | no |
| SPS w/o ret, std Re-Eval | 1 | yes | no | no |
| SPS w/ ret, no Re-Eval | 2 (both periodic) | no | no | no |
| SPS w/ ret, std Re-Eval | 2 (both periodic) | yes | no | no |
| DIT w/o Re-Eval, no Pre-Emp | 2 for the first packet, then 1 | no | yes | no |
| DIT w/ Re-Eval and Pre-Emp | 2 for the first packet, then 1 | yes | yes | yes |

As in the paper (its footnote 1), pre-emption is only active in the DIT scheme.

### 1.3 What had to change in WiLabV2Xsim

**Branch.** The repository has an `Octave-version` branch (August 2022), but it predates
re-evaluation. `main` (612bf26, November 2025) has the standard re-evaluation and is the base here.
Neither has DIT or pre-emption, and the repository has no scripts for this paper.

**Octave port** (Octave 10.3, conda-forge, inside the project):

* `constants.m`: a constant defined from other constants of its own class sends Octave into an
  infinite recursion; the values are written out.
* `searchParamInCfgFile.m`: Octave's `textscan` with comment handling dropped lines of the config
  file (the 20 MHz bandwidth among them); replaced by a line reader.
* `getseed.m`, `outputToFiles.m`: MATLAB `string` syntax (`"a" + b`).
* `round(x, n)` (not in Octave) → `roundDigits(x, n)`, in 18 files.
* `x'++y` read as an increment by Octave → `x'+y`.
* `db2pow`, `pow2db`, `unifrnd`, `exprnd` from MATLAB toolboxes: small equivalents in
  `octave_compat/` instead of the Octave Forge `statistics` package, whose current release needs
  Octave 11.
* On macOS 26 the conda-forge Octave cannot locate itself and recurses at start-up;
  `octave.sh` sets `OCTAVE_HOME`.

**Upstream bug fixed.** With re-evaluation and one copy per packet, the code that keeps the
unchanged resource read the second copy's entry, which does not exist
(`BRtoChange(1,2)`): the "SPS w/o ret, std Re-Eval" scheme stopped with an index error at its first
resource change. MATLAB would stop the same way.

**DIT and pre-emption** (new, from the paper's Section III):

1. At every SPS reselection (counter at 0) the UE gets two resources for the first packet, the
   second within 32 slots of the first, with the same selection procedure as a blind retransmission.
2. The SCI of the first transmission does not reserve its own resource for the next period
   (RRI 0) and announces the second resource. That announcement is marked as **high priority**:
   WiLabV2Xsim has no priority levels, so this mark stands for the paper's "priority of the first
   transmission at a high level".
3. Standard re-evaluation (when on) checks both resources 3 slots before the first transmission.
4. **Pre-emption check**, 3 slots before the second transmission: the UE rebuilds Sa as in a
   selection; the second resource is left only if it is no longer in Sa **and** a DIT first
   transmission announced it (the priority condition). It is then replaced by a random resource of
   Sa, before the PDB of the packet.
5. After the second transmission the first resource is released and the second one becomes the
   periodic resource (`cv2xNumberOfReplicas` back to 1).
6. If no second resource fits in the window, that sequence runs as plain SPS.

Pre-emption by an ordinary periodic reservation is not modelled: the paper uses pre-emption only
for the DIT announcements, and all ordinary packets have the same priority.

## 2. PRR against distance (paper Fig. 3)

Packet reception ratio per 10 m bin: receivers that decoded the packet over the receivers at that
distance. Line: mean over the seeds; band: 95 % interval over the seeds, computed on the logit scale
so it stays within [0, 1]. Same axes as the paper (20 to 200 m, PRR 0.85 to 1).

In [ ]:
def read_prr(d):
    f = os.path.join(d, "packet_reception_ratio_1_5G.xls")
    t = pd.read_csv(f, sep="\t", header=None, names=["dist", "correct", "error", "blocked", "total", "prr"])
    return t

def logit(p): return np.log(p / (1 - p))
def expit(z): return 1 / (1 + np.exp(-z))

prr = []
for _, r in R.iterrows():
    t = read_prr(r.dir).assign(scheme=r.scheme, rho=r.rho, seed=r.seed)
    prr.append(t)
prr = pd.concat(prr, ignore_index=True)
prr = prr[prr.total > 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
for ax, rho in zip(axes, [50, 200]):
    for key, label, col, ls in SCHEMES:
        part = prr[(prr.rho == rho) & (prr.scheme == key)]
        if part.empty:
            continue
        floor = 0.5 / part.total
        part = part.assign(lv=logit(part.prr.clip(lower=floor, upper=1 - floor)))
        st = part.groupby("dist").agg(m=("prr", "mean"), lm=("lv", "mean"), ls_=("lv", "std"), n=("lv", "count"))
        h = (st.n.map(T95).fillna(1.96) * st.ls_ / np.sqrt(st.n)).fillna(0)
        ax.plot(st.index, st.m, color=col, ls=ls, lw=2, label=label)
        ax.fill_between(st.index, expit(st.lm - h), expit(st.lm + h), color=col, alpha=0.15, lw=0)
    ax.set_xlim(20, 200); ax.set_ylim(0.85, 1.0)
    ax.set_xlabel("Distance [m]"); ax.set_title("%d v/km" % rho)
axes[0].set_ylabel("PRR"); axes[0].legend(fontsize=8.5, loc="lower left")
plt.tight_layout(); plt.show()

## 3. Wireless blind spot probability within 150 m (paper Fig. 4)

Probability that no packet is received from a UE within 150 m for at least the given duration
(WiLabV2Xsim's wireless blind spot counter, Bazzi et al., IEEE TVT 2020). Line: all seeds pooled
(blind events over all events); band: lowest to highest seed. Log-log axes as in the paper.

In [ ]:
def read_wbs(d):
    f = os.path.join(d, "wireless_blind_spot_1_5G.xls")
    return pd.read_csv(f, sep="\t", header=None,
                       names=["dur", "b150", "n150", "p150", "b200", "n200", "p200"])

wbs = []
for _, r in R.iterrows():
    wbs.append(read_wbs(r.dir).assign(scheme=r.scheme, rho=r.rho, seed=r.seed))
wbs = pd.concat(wbs, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
for ax, rho in zip(axes, [50, 200]):
    for key, label, col, ls in SCHEMES:
        part = wbs[(wbs.rho == rho) & (wbs.scheme == key)]
        if part.empty:
            continue
        pooled = part.groupby("dur").apply(lambda g: g.b150.sum() / (g.b150.sum() + g.n150.sum()))
        lo = part.groupby("dur").p150.min(); hi = part.groupby("dur").p150.max()
        pooled = pooled.where(pooled > 0)
        ax.plot(pooled.index, pooled.values, color=col, ls=ls, lw=2, label=label)
        ax.fill_between(lo.index, lo.where(lo > 0, np.nan), hi.where(hi > 0, np.nan), color=col, alpha=0.12, lw=0)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(0.1, 10); ax.set_ylim(1e-5, 1)
    ax.set_xlabel("Wireless Blind Spot Duration [s]"); ax.set_title("%d v/km" % rho)
axes[0].set_ylabel("Probability"); axes[0].legend(fontsize=8.5, loc="lower left")
plt.tight_layout(); plt.show()

## 4. The paper's claims, checked

Each claim of the paper's Section IV-B against the reproduced numbers (mean over the seeds, ±
95 % interval). PRR at 50, 100 and 150 m; WBSP at 0.5 s and 1 s within 150 m.

In [ ]:
def ci(v):
    v = np.asarray(v, float)
    if len(v) == 0:
        return np.nan, np.nan
    if len(v) < 2:
        return v.mean(), np.nan
    return v.mean(), T95.get(len(v), 1.96) * v.std(ddof=1) / np.sqrt(len(v))

def prr_at(key, rho, dist):
    v = prr[(prr.scheme == key) & (prr.rho == rho) & (prr.dist == dist)].prr
    return ci(v)

def wbs_at(key, rho, dur):
    part = wbs[(wbs.scheme == key) & (wbs.rho == rho) & np.isclose(wbs.dur, dur)]
    return part.b150.sum() / max(part.b150.sum() + part.n150.sum(), 1)

rows = []
for rho in (50, 200):
    for key, label, _, _ in SCHEMES:
        row = {"density": rho, "scheme": label}
        for d in (50, 100, 150):
            m, h = prr_at(key, rho, d)
            row["PRR %d m" % d] = "%.4f ± %.4f" % (m, h) if not np.isnan(m) else ""
        for s in (0.5, 1.0):
            row["WBSP %.1f s" % s] = "%.2e" % wbs_at(key, rho, s)
        rows.append(row)
summary = pd.DataFrame(rows).set_index(["density", "scheme"])
summary

In [ ]:
def gap(a, b, rho, dist):
    """PRR(a) - PRR(b) at a distance, paired by seed, in points, with its 95 % interval."""
    pa = prr[(prr.scheme == a) & (prr.rho == rho) & (prr.dist == dist)].set_index("seed").prr
    pb = prr[(prr.scheme == b) & (prr.rho == rho) & (prr.dist == dist)].set_index("seed").prr
    d = (pa - pb).dropna() * 100
    m, h = ci(d)
    return "%+.2f ± %.2f" % (m, h) if len(d) else ""

checks = []
for rho in (50, 200):
    for dist in (50, 100):
        checks.append({"density": rho, "distance": dist,
            "re-evaluation, no ret (paper: no effect)": gap("sps_noret_reeval", "sps_noret_noreeval", rho, dist),
            "retransmission (paper: + at 50, − at 200)": gap("sps_ret_noreeval", "sps_noret_noreeval", rho, dist),
            "re-evaluation, with ret (paper: marginal +)": gap("sps_ret_reeval", "sps_ret_noreeval", rho, dist),
            "DIT + re-eval + pre-emption vs legacy (paper: +)": gap("dit_reeval_preempt", "sps_noret_noreeval", rho, dist),
            "DIT without re-eval vs legacy (paper: marginal)": gap("dit_noreeval", "sps_noret_noreeval", rho, dist)})
pd.DataFrame(checks).set_index(["density", "distance"]).rename_axis(columns="Δ PRR [points]")

In [ ]:
ratio = []
for rho in (50, 200):
    legacy = wbs_at("sps_noret_noreeval", rho, 1.0)
    dit = wbs_at("dit_reeval_preempt", rho, 1.0)
    ratio.append({"density": rho, "WBSP 1 s, legacy SPS": "%.2e" % legacy, "WBSP 1 s, DIT": "%.2e" % dit,
                  "reduction": "%.1f×" % (legacy / dit) if dit > 0 else "DIT: no blind spot of 1 s",
                  "paper": "more than 10×" if rho == 50 else "about 50 %"})
pd.DataFrame(ratio).set_index("density")

## 5. How often the DIT pre-emption acts

Per run: pre-emption checks (one per DIT first packet whose second resource is still announced)
and resources pre-empted, from `dit_preemption.txt`.

In [ ]:
rows = []
for _, r in R[R.scheme == "dit_reeval_preempt"].iterrows():
    f = os.path.join(r.dir, "dit_preemption.txt")
    if os.path.exists(f):
        v = dict(line.split() for line in open(f))
        rows.append({"density": r.rho, "seed": r.seed, "checks": int(v["checks"]), "pre-empted": int(v["preempted"])})
pe = pd.DataFrame(rows)
if len(pe):
    pe["share pre-empted [%]"] = (100 * pe["pre-empted"] / pe.checks).round(2)
pe.sort_values(["density", "seed"]) if len(pe) else "no DIT run with pre-emption yet"

## 6. Run cost

Wall-clock time of each run on ease-rtx (one core per run, 36 runs on 32 cores).

In [ ]:
R2 = R.assign(seconds=[int(open(os.path.join(d, "done")).read().strip() or 0) for d in R.dir])
(R2.groupby(["rho", "scheme"]).seconds.mean() / 60).round(1).unstack("rho").rename(index=LABEL).rename_axis(columns="minutes per run")

## 7. WiLabV2Xsim, MoReV2X and 5G-LENA

From the code of the three simulators as used in this project: WiLabV2Xsim `main` 612bf26 (with
the patch above), MoReV2X on ns-3.22 (this repository, branch `feature/qos-evaluation`), and
5G-LENA `nr` v2x-1.1 on ns-3.42 (the `groupcast-adaptation-ns3` tree).

| | WiLabV2Xsim | MoReV2X | 5G-LENA (NR V2X) |
|---|---|---|---|
| Platform | MATLAB (here Octave 10.3) | ns-3.22, C++ | ns-3.42, C++ |
| Protocol stack | MAC resource allocation and PHY abstraction only | MAC and PHY of Mode 2, RLC/PDCP from the LTE module | full NR stack (SDAP to PHY) |
| Time and resources | one TTI at a time over a grid of *beacon resources*: every packet of a run has the same size and the same number of subchannels | per slot, per subchannel; packets of any size within a slot | per slot, per subchannel, per symbol (PSCCH, PSSCH, PSFCH) |
| Reception model | SINR threshold (3.6 dB here) or PER-vs-SINR curves | BLER-vs-SINR curves per MCS, LOS/NLOS, speed | EESM / mutual-information link-to-system mapping |
| Channel | WINNER+ B1 and other path-loss models, correlated shadowing, optional Rayleigh fading | TR 37.885 highway (LOS / NLOSv), plus the TR 37.885 urban grid of this fork | 3GPP TR 38.901 / 37.885 spatial channel through the ns-3 spectrum model |
| Mobility | built-in highway and urban generators, or a trace file | SUMO traces (this fork) or the built-in highway | ns-3 mobility, SUMO traces in the groupcast tree |
| Sensing and selection | SCI decoding with an RSRP threshold, 20 % rule, 3 dB steps | TS 38.214 §8.1.4 steps 1–7, priority-pair thresholds (this fork) | TS 38.214 §8.1.4 sensing |
| SPS and dynamic scheduling | both, reselection counter and keep probability | both, per flow (this fork) | both (SPS and dynamic grants) |
| Re-evaluation | yes, once, 3 slots before the first transmission of a new selection | yes, at T3 = Tproc,1 before every not-yet-announced resource | **no** ("we do not support grant reevaluation/reselection") |
| Pre-emption | not upstream; added here for the DIT announcements only | yes, with SL priorities and the priority threshold (this fork) | **no** |
| Priorities | none | SL priority 1–8 per flow (this fork) | per logical channel |
| Retransmissions | blind, at most one, within 32 slots | blind, up to 3 copies, within 32 slots and the PDB (this fork) | blind and HARQ feedback on PSFCH |
| Traffic | periodic, 3GPP aperiodic, ETSI CAM; one flow per vehicle | periodic, aperiodic, variable size, several flows per UE (this fork) | ns-3 applications, several flows |
| Cast types | broadcast | broadcast; groupcast and unicast for platooning | broadcast, groupcast, unicast |
| Metrics built in | PRR vs distance, wireless blind spots, update delay, CBR | PDR, PIR, DT, AoI, causes, CBR (this fork) | ns-3 traces (PSSCH, PSCCH, PSFCH, MAC), PIR examples |
| Speed here | 50 v/km: about 10 s per simulated second on one core of ease-rtx | QoS sweep, 15 s runs: about 1.1 h on average, 1.6 h for the densest | not run in this comparison |

**What this means for comparing their results**

* WiLabV2Xsim is the fastest and the closest to the paper (its authors' tool), but its grid makes
  every packet the same size and its PHY is a threshold: it cannot show variable-size CPMs, several
  flows per UE or priorities without changes like the ones made here.
* MoReV2X follows the re-evaluation timing and pre-emption rules of TS 38.214 most closely of the
  three, and adds what WiLabV2Xsim lacks (variable sizes, priorities, several flows); its PHY
  uses BLER curves, so the same SINR does not give a hard yes/no as in WiLabV2Xsim.
* 5G-LENA has the most complete stack and channel, and HARQ feedback, but no re-evaluation or
  pre-emption: the questions of this paper and of notebooks 12 to 14 cannot be studied in it
  without implementing them first.
* The three simulators define their metrics differently (PRR per receiver within a range, PDR per
  attempt within 1000 m, blind spot vs disconnection time), so numbers compare within one
  simulator, and trends compare across them.

## 8. What the reproduction shows

Written from the ease-rtx sweep (36 runs: 6 schemes × 2 densities × 3 seeds; 60 s at
50 vehicles/km, 20 s at 200 vehicles/km). Numbers are means over the seeds; differences are paired
by seed, with their 95 % interval (section 4).

**Every claim of the paper reproduces.**

| Paper (Section IV-B) | Reproduced |
|---|---|
| Re-evaluation does nothing when each packet is sent once | +0.03 to +0.23 points of PRR at 50 and 100 m, all within the noise |
| Retransmissions help only at low density | +0.6 to +1.3 points at 50 vehicles/km; at 200 vehicles/km +3.2 points at 50 m but −9.4 points at 100 m (the extra copies congest the channel) |
| Re-evaluation brings a marginal gain with retransmissions | +0.05 to +1.0 points; significant only at 200 vehicles/km, 100 m |
| DIT with re-evaluation and pre-emption beats legacy SPS, most at short distance | +0.7 and +1.5 points at 50 and 100 m (50 vehicles/km); +5.0 and +5.4 points (200 vehicles/km) |
| Without re-evaluation, DIT gains little: the gain comes from the avoided collisions, not from the extra copy | +0.0 to +0.7 points, −0.9 at 200 vehicles/km and 100 m |
| 1 s blind spots: more than 10× fewer at low density, about 50 % fewer at high density | 15× fewer at 50 vehicles/km (8.1·10⁻⁴ → 5.4·10⁻⁵), 2.4× fewer at 200 vehicles/km (1.5·10⁻² → 6.1·10⁻³) |

**The curves match the paper's figures.** At 200 vehicles/km the PRR falls to 0.85 at about 93 m
for legacy SPS, 82 m with retransmissions and 113 m with DIT; read from the paper's Fig. 3b, about
95–100, 85 and 118 m. At 50 vehicles/km the retransmission curves stay on top, and DIT with
re-evaluation follows them up to about 130 m before falling between them and legacy SPS, as in
Fig. 3a. The blind-spot curves (Fig. 4) keep the paper's order at both densities: at 200
vehicles/km retransmissions give the longest blind spots and DIT the shortest.

**How DIT works in the runs.** The pre-emption check runs for nearly every new SPS sequence
(about 5 600 checks per 60 s run at 50 vehicles/km, 7 700 per 20 s run at 200 vehicles/km). It
moves the second resource in 0.4 to 0.6 % of the checks at 50 vehicles/km and 4 to 5 % at 200: each
of these is a persistent collision of the paper's Case 3, avoided before it started. Removing that
small number of collisions, which would otherwise repeat for the whole reselection period, is
what shortens the blind spots.

**Limits of this reproduction.**

* DIT and pre-emption are implemented here from the paper's description; the authors' own code is
  not public. Priorities are reduced to one mark ("announced by a DIT first transmission"),
  because WiLabV2Xsim has no priority levels.
* The paper gives no simulation length or number of runs. 60 s and 20 s with 3 seeds give tight
  PRR intervals, but the blind-spot curves below 10⁻⁴ rest on few events.
* WiLabV2Xsim ran in Octave with the patch of section 1.3; the Octave port changes no model, only
  syntax and helper functions, and fixes one upstream bug that stopped one scheme.

**Relation to the MoReV2X results (notebooks 12 to 14).** Both simulators agree that
re-evaluation on its own changes little for periodic traffic. In MoReV2X it gave +0.7 to +1.5
points on the dense highway for flows that use reserved (SB-SPS) resources, and +4 points for the
aperiodic CPMs. The DIT idea addresses what neither re-evaluation nor plain pre-emption could fix
there: persistent collisions between SPS reservations that sensing never saw announced. It could be
tested in MoReV2X with its priority-aware pre-emption, where the first transmission's priority is
an SL priority rather than a mark.